# Code was run on Colab Pro

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import collections
import torch.optim as optim
from torch.optim import Optimizer
import time
import matplotlib.pyplot as plt

from AdamW          import AdamW
from utils          import utility, misreportUtility, misreportOptimization, trueUtility, loss
from networks       import AdditiveMechanism, Misreports,AllocationNet,PaymentNet
from restrictedAdam import Adam 

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.cuda.set_device(2)

# Set Random Seed 

In [3]:
# Initializing seeds
torch.manual_seed(2)
np.random.seed(2)

# Testing Function

In [4]:
def test(nBatch, nbrInit, R, gamma=0.001, minimum=0, maximum=1):
    
    """ This function computes the regret and payment of mechanism on a test set of size nBatch
        The optimal misreport is computed by optimizing the utility function (not by using the Misreport network)
        for R gradient steps (of stepsize gamma) and starting from nbrInit initialization, we only keep the best misreport
        To compute the regret we evaluate the mechanism at the misreport and compare to the valuation
        minimum and maximum indicate the range of the valuations
    """
    
    true = np.random.rand(nBatch,nAgent,nObject)

    localMisreports     = np.random.rand(nBatch,nbrInit,nAgent,nObject)
    batchMisreports     = torch.tensor(localMisreports).float().to(device)
    batchTrueValuations = torch.tensor(true).float().to(device)
    batchMisreports.requires_grad = True
    
    opt = Adam([batchMisreports], lr=gamma)
    
    for k in range(R):
        advU         = misreportUtility(mechanism,batchTrueValuations,batchMisreports)
        los          =  -1*torch.mean(advU).to(device)
        los.backward()
        opt.step(restricted= True, min=minimum, max=maximum)
        opt.zero_grad()
    
    misReportUtilityMax  = torch.max(advU, dim =1)[0]
    mechanism.zero_grad()
    allocation, payment = mechanism(batchTrueValuations)
    regret = F.relu(misReportUtilityMax -utility(batchTrueValuations, allocation, payment))
    mregret= torch.sum(torch.mean(regret, dim=0)).to(device)
    mregret= float(mregret.cpu().detach().numpy())

    with torch.no_grad():
        l,rMean,p = loss(payment, regret)

    testRegret.append(mregret)
    testPayment.append(float(p.detach().cpu().numpy() ))
    testOptimal.append(float((-l).detach().cpu().numpy())**2)
    print("Total regret: ",'{0:.5f}'.format(mregret), "Average regret per bidder: ",'{0:.5f}'.format(mregret/nAgent), " Optimal Revenue: ",'{0:.3f}'.format(float((-l).detach().cpu().numpy())**2), " payment: ",'{0:.3f}'.format(float(p.detach().cpu().numpy() )))

# Initializing Networks

In [5]:
nAgent   = 3
nObject  = 10

# Parameters for the mechanism (payment and allocation network)
nLayersAllocation   = 7
nLayersPayment      = 7
widthAllocation     = 100
widthPayment        = 100

# Parameters for the misreport network
nLayersMisreport    = 7
widthMisreport      = 100

gamma              = 0.001 
testBatch          = 10000

nExperiments       = 200000
batchSize          = 500
nbrBatches         = int(nExperiments/batchSize)



alloc_net = AllocationNet(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
opt_alloc = AdamW(alloc_net.parameters(), lr=1e-3)

pay_net   = PaymentNet(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
opt_pay   = AdamW(pay_net.parameters(),   lr=1e-3)

mechanism            = AdditiveMechanism(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
optimizerMechanism   = AdamW(mechanism.parameters(), lr=0.001)

misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
optimizerMisreport   = AdamW(misreport.parameters(), lr=0.001)

In [6]:
testRegret    = []
testMaxRegret = []
testPayment   = []
testOptimal   = []
testTime      = []
testIteration = [0]

# range of valuations
minimum            = 0
maximum            = 1

In [7]:
@torch.no_grad()
def myerson_itemwise_allocation_payment(values, reserve=0.5):
    """
    values:  (B, A, O)  估值矩阵
    reserve: 保留价
    return:  alloc (B, A, O), pay_myr (B, A)
    """
    B, A, O = values.shape
    # 逐物品取 top-2 出价
    top2 = values.topk(k=2, dim=1)
    v1, idx1 = top2.values[:, 0, :], top2.indices[:, 0, :]  # 最高价及其索引
    v2 = top2.values[:, 1, :]                               # 第二高价

    # 判断是否超过保留价
    win = (v1 >= reserve).float()        # (B,O)
    price = torch.maximum(v2, torch.full_like(v2, reserve)) * win

    # 构造 one-hot 分配矩阵
    alloc = torch.zeros(B, A, O, device=values.device)
    alloc.scatter_(1, idx1.unsqueeze(1), win.unsqueeze(1))

    # 计算每个代理的总支付
    pay_myr = alloc * price.unsqueeze(1)  # (B,A)
    return alloc, pay_myr

# Training

In [8]:
R=10000
reserve=0.5
print("Train AllocationNet with Myerson supervision")
for t in range(1,60*nbrBatches+1):
    # 随机估值
    values = torch.rand(batchSize, nAgent, nObject, device=device)

    # 计算 Myerson 标签
    with torch.no_grad():
        alloc_myr, pay_myr = myerson_itemwise_allocation_payment(values, reserve=reserve)

    # 网络输出
    alloc_pred = alloc_net(values)

    # 监督损失
    loss_alloc = F.mse_loss(alloc_pred, alloc_myr)

    opt_alloc.zero_grad()
    loss_alloc.backward()
    opt_alloc.step()
    if t % (2*nbrBatches)==0 :
        print(f"loss={loss_alloc.item():.6f}")

Train AllocationNet with Myerson supervision
loss=0.025551
loss=0.021756
loss=0.017896
loss=0.015378
loss=0.013558
loss=0.012158
loss=0.012116
loss=0.010976
loss=0.009483
loss=0.010712
loss=0.009194
loss=0.009637
loss=0.008595
loss=0.009199
loss=0.009236
loss=0.008351
loss=0.007924
loss=0.007334
loss=0.008152
loss=0.008018
loss=0.007633
loss=0.007710
loss=0.006521
loss=0.007349
loss=0.008675
loss=0.008170
loss=0.007499
loss=0.007031
loss=0.006207
loss=0.006788


In [9]:
print("Train PaymentNet with Myerson supervision")

for t in range(1,60*nbrBatches+1):
    values = torch.rand(batchSize, nAgent, nObject, device=device)

    with torch.no_grad():
        alloc_myr, pay_myr = myerson_itemwise_allocation_payment(values, reserve=reserve)

    payments_pred = pay_net(values, alloc_myr)

    loss_pay = F.mse_loss(payments_pred, pay_myr)

    opt_pay.zero_grad()
    loss_pay.backward()
    opt_pay.step()
    if t % (2*nbrBatches)==0 :
        print(f"loss={loss_pay.item():.6f}")

Train PaymentNet with Myerson supervision
loss=0.001385
loss=0.000769
loss=0.000413
loss=0.000315
loss=0.000256
loss=0.000230
loss=0.000184
loss=0.000172
loss=0.000145
loss=0.000143
loss=0.000124
loss=0.000117
loss=0.000110
loss=0.000104
loss=0.000093
loss=0.000086
loss=0.000078
loss=0.000088
loss=0.000074
loss=0.000064
loss=0.000065
loss=0.000063
loss=0.000059
loss=0.000056
loss=0.000054
loss=0.000055
loss=0.000048
loss=0.000047
loss=0.000045
loss=0.000050


In [10]:
duration   = 0
R          = 100

i=0
mechanism            = AdditiveMechanism(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)

mechanism.alloc_net.load_state_dict(alloc_net.state_dict())
mechanism.payment_net.load_state_dict(pay_net.state_dict())
optimizerMechanism   = AdamW(mechanism.parameters(), lr=0.001)

print("Initial Test")
test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

for t in range(1,60*nbrBatches+1):
    
    # Reinitialize Misreport network periodically at the beginning of training
    if (t%(2*nbrBatches) ==1):
      if   t< 20*nbrBatches+2 :
    
        misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
        optimizerMisreport   = AdamW(misreport.parameters(), lr=0.001)

    batchTrueValuations = torch.tensor(np.random.rand(batchSize,nAgent,nObject)).float().to(device)
    
    # Optimize Misreport Network for R steps
    for k in range(R):
  
        misreports          = misreport(batchTrueValuations).unsqueeze(1)
        mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)
        mLoss               = torch.sum(torch.mean(-mUtility,dim=0))

        optimizerMisreport.zero_grad()
        mLoss.backward()
        optimizerMisreport.step()

    
    # Optimize Mechanism network for one step
    misreports          = misreport(batchTrueValuations).unsqueeze(1)
    mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)

    allocation, payment = mechanism(batchTrueValuations)

    regret     = F.relu(mUtility -utility(batchTrueValuations, allocation, payment))
    l,rMean,p = loss(payment, regret)
        
    optimizerMechanism.zero_grad()

    l.backward()

    optimizerMechanism.step()
    
    # Test mechanism periodically
    if t % (2*nbrBatches)==0 :
        print("Batch: ", 2*int(t/(2*nbrBatches)))
        testTime.append(duration)
        testIteration.append(t/nbrBatches)
        test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

Initial Test


/home/wkw/ysy/women (1)/restrictedAdam.py:103: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha) (Triggered internally at  ../torch/csrc/utils/python_arg_parser.cpp:1050.)
  exp_avg.mul_(beta1).add_(1 - beta1, grad)


Total regret:  0.10648 Average regret per bidder:  0.03549  Optimal Revenue:  3.389  payment:  5.170
Batch:  2
Total regret:  0.01637 Average regret per bidder:  0.00546  Optimal Revenue:  4.847  payment:  5.503
Batch:  4
Total regret:  0.01653 Average regret per bidder:  0.00551  Optimal Revenue:  4.931  payment:  5.596
Batch:  6
Total regret:  0.01351 Average regret per bidder:  0.00450  Optimal Revenue:  5.061  payment:  5.662
Batch:  8
Total regret:  0.01157 Average regret per bidder:  0.00386  Optimal Revenue:  5.097  payment:  5.649
Batch:  10
Total regret:  0.01499 Average regret per bidder:  0.00500  Optimal Revenue:  5.112  payment:  5.752
Batch:  12
Total regret:  0.01133 Average regret per bidder:  0.00378  Optimal Revenue:  5.112  payment:  5.658
Batch:  14
Total regret:  0.01002 Average regret per bidder:  0.00334  Optimal Revenue:  5.137  payment:  5.648
Batch:  16
Total regret:  0.01625 Average regret per bidder:  0.00542  Optimal Revenue:  5.042  payment:  5.708
Batch: 

# Testing

In [11]:
for i in range(200):
    test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

Total regret:  0.00433 Average regret per bidder:  0.00144  Optimal Revenue:  5.451  payment:  5.783
Total regret:  0.00561 Average regret per bidder:  0.00187  Optimal Revenue:  5.178  payment:  5.551
Total regret:  0.00528 Average regret per bidder:  0.00176  Optimal Revenue:  5.314  payment:  5.679
Total regret:  0.00417 Average regret per bidder:  0.00139  Optimal Revenue:  5.436  payment:  5.761
Total regret:  0.00438 Average regret per bidder:  0.00146  Optimal Revenue:  5.526  payment:  5.863
Total regret:  0.00428 Average regret per bidder:  0.00143  Optimal Revenue:  5.368  payment:  5.695
Total regret:  0.00396 Average regret per bidder:  0.00132  Optimal Revenue:  5.357  payment:  5.671
Total regret:  0.00414 Average regret per bidder:  0.00138  Optimal Revenue:  5.470  payment:  5.795
Total regret:  0.00538 Average regret per bidder:  0.00179  Optimal Revenue:  5.345  payment:  5.716
Total regret:  0.00442 Average regret per bidder:  0.00147  Optimal Revenue:  5.397  paymen

Total regret:  0.00488 Average regret per bidder:  0.00163  Optimal Revenue:  5.440  payment:  5.794
Total regret:  0.00321 Average regret per bidder:  0.00107  Optimal Revenue:  5.600  payment:  5.887
Total regret:  0.00462 Average regret per bidder:  0.00154  Optimal Revenue:  5.471  payment:  5.816
Total regret:  0.00430 Average regret per bidder:  0.00143  Optimal Revenue:  5.385  payment:  5.714
Total regret:  0.00382 Average regret per bidder:  0.00127  Optimal Revenue:  5.433  payment:  5.743
Total regret:  0.00450 Average regret per bidder:  0.00150  Optimal Revenue:  5.519  payment:  5.860
Total regret:  0.00524 Average regret per bidder:  0.00175  Optimal Revenue:  5.378  payment:  5.745
Total regret:  0.00529 Average regret per bidder:  0.00176  Optimal Revenue:  5.364  payment:  5.731
Total regret:  0.00499 Average regret per bidder:  0.00166  Optimal Revenue:  5.313  payment:  5.668
Total regret:  0.00458 Average regret per bidder:  0.00153  Optimal Revenue:  5.527  paymen

Total regret:  0.00421 Average regret per bidder:  0.00140  Optimal Revenue:  5.388  payment:  5.713
Total regret:  0.00466 Average regret per bidder:  0.00155  Optimal Revenue:  5.429  payment:  5.775
Total regret:  0.00522 Average regret per bidder:  0.00174  Optimal Revenue:  5.438  payment:  5.805
Total regret:  0.00464 Average regret per bidder:  0.00155  Optimal Revenue:  5.318  payment:  5.658
Total regret:  0.00545 Average regret per bidder:  0.00182  Optimal Revenue:  5.454  payment:  5.831
Total regret:  0.00462 Average regret per bidder:  0.00154  Optimal Revenue:  5.365  payment:  5.706
Total regret:  0.00367 Average regret per bidder:  0.00122  Optimal Revenue:  5.459  payment:  5.763
Total regret:  0.00353 Average regret per bidder:  0.00118  Optimal Revenue:  5.429  payment:  5.726
Total regret:  0.00470 Average regret per bidder:  0.00157  Optimal Revenue:  5.290  payment:  5.632
Total regret:  0.00634 Average regret per bidder:  0.00211  Optimal Revenue:  5.332  paymen

In [12]:
totalregret = np.mean(np.array(testRegret[-200:]))
revenue     = np.mean(np.array(testPayment[-200:]))
print("Final Result")
print("Total Regret = ", '{0:.5f}'.format(totalregret), "Average regret per bidder: ",'{0:.5f}'.format(totalregret/nAgent), " Optimal Revenue: ",'{0:.3f}'.format(float(np.sqrt(revenue)-np.sqrt(totalregret))**2), " payment: ",'{0:.3f}'.format(revenue))

Final Result
Total Regret =  0.00456 Average regret per bidder:  0.00152  Optimal Revenue:  5.426  payment:  5.745


In [13]:
stdregret = np.std(np.array(testRegret[-200:]))
stdrevenue= np.std(np.array(testPayment[-200:]))
print("std Regret = ", '{0:.5f}'.format(stdregret), "std regret per bidder: ",'{0:.5f}'.format(stdregret/nAgent), " std payment: ",'{0:.3f}'.format(stdrevenue))

std Regret =  0.00074 std regret per bidder:  0.00025  std payment:  0.076


In [15]:
torch.save(mechanism,"310stage2.pt")